## tl;dr
用户确认旧/新记录均无风、相同设定高度。B10/B11各用两次同配对Hover等权平均得到自己的三段曲线；上分界没有固定80%。固定80%的近期单次方案另存为对照。

## Context & Methods
### Key Assumptions
95%→20%；每段至少10个百分点/20秒。新旧记录从95%对齐，在每个相同SOC取首次到达耗时的算术平均，再拟合三段。构造均值不是额外真实实验。
这个核查本是模型的 companion。所有普通 Python 单元逐格运行；未验证 Jupyter 内核执行。

In [1]:
from pathlib import Path
import json, hashlib, sys
import pandas as pd
root=Path('/Users/alexmason/Downloads/drone_experiment')
sys.path.insert(0,str(root))
from output_py.revise_b10_b11_calibration import review_sources, pooled_curve
from output_py.search_bideal_boundaries import fit
folder=Path('/Users/alexmason/Downloads/drone_experiment/analysis_results/battery_normalization_pooled_v2_20260909')
model=json.loads((folder/'model.json').read_text())
alternative=json.loads((folder/'fixed_80_sensitivity_model.json').read_text())
traces, profiles, individual=review_sources()
print('Reviewed hover records:',len(profiles))

Reviewed hover records: 11


## Data
实时重读原始记录及 SHA256；只把当前同一电池/无人机配对用于新旧平均对照。

In [2]:
print(pd.DataFrame(profiles)[['battery_id','drone_id','run_id','use']].to_string(index=False))

battery_id drone_id                         run_id                              use
       B10  drone_1                20260830_174908                excluded_no_hover
       B10  drone_1                20260830_174922        excluded_incomplete_95_20
       B11  drone_1                20260513_143201       primary_pooled_calibration
       B11  drone_1                20260514_135933                excluded_no_hover
       B11  drone_1                20260514_151213                excluded_no_hover
       B11  drone_1                20260906_164629                excluded_no_hover
       B11  drone_1                20260906_164725       primary_pooled_calibration
       B10  drone_2                20260513_180556       primary_pooled_calibration
       B10  drone_2                20260906_172945                excluded_no_hover
       B10  drone_2                20260906_173013       primary_pooled_calibration
       B10  drone_3 batterytest_10_20260506_115206 different_drone_sensitivi

## Results
重新计算对照与主候选，核对保存的两个电池系数；不覆盖模型。

In [3]:
for battery in ['B10','B11']:
    item=alternative['batteries'][battery]
    trace=next(t for t in traces if t['battery_id']==battery and t['run_id']==item['run_id'])
    result=fit(trace,80,int(item['boundaries_soc'][2]),20)
    assert abs(result['rmse']-item['in_sample_rmse_pp'])<1e-10
    print('Alternative',battery,item['boundaries_soc'],'RMSE',result['rmse'])

Alternative B10 [95.0, 80.0, 56.0, 20.0] RMSE 1.2678976971902565
Alternative B11 [95.0, 80.0, 56.0, 20.0] RMSE 1.469413822548195


In [4]:
for battery,drone in [('B10','drone_2'),('B11','drone_1')]:
    selected=[t for t in traces if t['battery_id']==battery and t['drone']==drone]
    curve, _, grid, _, _=pooled_curve(selected)
    assert list(curve.boundaries)==model['batteries'][battery]['boundaries_soc']
    assert all(abs(a-b)<1e-10 for a,b in zip(curve.rates_pp_min,model['batteries'][battery]['rates_pp_min']))
    print('Primary old/new mean:',battery,curve.boundaries,'approximation RMSE',float(grid.iloc[0].rmse_pp))

Primary old/new mean: B10 (95.0, 85.0, 56.0, 20.0) approximation RMSE 1.1534262304676997
Primary old/new mean: B11 (95.0, 82.0, 51.0, 20.0) approximation RMSE 1.0142768448729433


## Takeaways
主候选用重复实验的平均特性，不是只追求最新一次最低误差。B10为85/56、B11为82/51；B11新旧差异仍较大，不能宣称校准可靠性已获独立验证。原始数据、上一版及飞行逻辑未改。